In [1]:
import pandas as pd
import zipfile
import os
import numpy as np

In [2]:
path = 'Datasets/AQI_zipfiles'

def read_clean_AQI(path: str) -> pd.DataFrame:
  # unzips and reads in each zipfile in a folder, reads each zipfile into a csv,
  # merges them into one dataframe and cleans up columns
  dict_dataframes = {}
  for root, dirs, files in os.walk(path):
    for file in files:
      file_name = file.split('.')[0]
      zf = zipfile.ZipFile(f'{path}/{file}')
      df = pd.read_csv(zf.open(f'{file_name}.csv'))
      dict_dataframes[file_name] = df


  df = pd.concat(dict_dataframes.values(),ignore_index=True)
  return df

df = read_clean_AQI(path)
df['Percent Moderate Days'] = df['Moderate Days'] / df['Days with AQI']

df.describe()
df.to_csv('Datasets/AQI_merged.csv')

df.head()

,State,County,Year,Days with AQI,Good Days,Moderate Days,Unhealthy for Sensitive Groups Days,Unhealthy Days,Very Unhealthy Days,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10,Percent Moderate Days
0,Alabama,Calhoun,1996,53,53,0,0,0,0,0,37,25,17,0,0,0,0,53,0.000000
1,Alabama,Clay,1996,242,157,66,17,2,0,0,161,93,45,0,0,242,0,0,0.272727
2,Alabama,Colbert,1996,54,54,0,0,0,0,0,44,27,14,0,0,0,0,54,0.000000
3,Alabama,DeKalb,1996,225,147,56,16,6,0,0,182,100,41,0,0,207,0,18,0.248889
4,Alabama,Elmore,1996,236,164,62,9,1,0,0,174,84,44,0,0,236,0,0,0.262712


In [3]:
# print list of states to clean up
df.State.unique()

array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Canada',
       'Colorado', 'Connecticut', 'Country Of Mexico', 'Delaware',
       'District Of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho',
       'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana',
       'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota',
       'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
       'New Hampshire', 'New Jersey', 'New Mexico', 'New York',
       'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon',
       'Pennsylvania', 'Puerto Rico', 'Rhode Island', 'South Carolina',
       'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont',
       'Virgin Islands', 'Virginia', 'Washington', 'West Virginia',
       'Wisconsin', 'Wyoming', 'Guam'], dtype=object)

In [4]:
# remove non-US states from state
df = df[~df['State'].isin(['Canada','Guam','Virgin Islands','Country Of Mexico', 'Puerto Rico'])]
print(f'There are {len(df.State.unique())} states in the dataset.')
df.State.unique()

There are 51 states in the dataset.


array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
       'Colorado', 'Connecticut', 'Delaware', 'District Of Columbia',
       'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
       'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
       'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
       'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
       'New Jersey', 'New Mexico', 'New York', 'North Carolina',
       'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
       'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee',
       'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

In [5]:
df.sample(10)

,State,County,Year,Days with AQI,Good Days,Moderate Days,Unhealthy for Sensitive Groups Days,Unhealthy Days,Very Unhealthy Days,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10,Percent Moderate Days
22957,Wyoming,Platte,2013,58,56,2,0,0,0,0,59,35,14,0,0,0,0,58,0.034483
29029,New York,Onondaga,2011,365,318,45,2,0,0,0,119,53,34,5,0,296,64,0,0.123288
40819,Illinois,Peoria,1998,365,297,58,9,1,0,0,154,67,39,126,0,232,0,7,0.158904
15186,Missouri,Jasper,2008,365,351,14,0,0,0,0,93,40,19,0,0,0,0,365,0.038356
24056,Arizona,Santa Cruz,2006,333,136,154,39,4,0,0,184,108,55,0,0,0,39,294,0.462462
25296,Florida,St. Lucie,2010,223,134,89,0,0,0,0,93,58,47,0,0,43,180,0,0.399103
26933,Ohio,Madison,2004,214,150,52,12,0,0,0,129,87,44,0,0,214,0,0,0.242991
33091,Kentucky,Hancock,2014,237,198,38,1,0,0,0,105,58,41,0,0,237,0,0,0.160338
12329,Oklahoma,Creek,2019,356,331,24,1,0,0,0,101,48,36,0,0,356,0,0,0.067416
21745,Texas,Harrison,2007,365,153,209,3,0,0,0,132,70,53,0,0,87,278,0,0.572603


In [6]:
# filter dataframe to only include counties with at least 50 days with AQI that year to remove small sample sizes
#county_df_with_fifty_days = df[df['Days with AQI'] <= 1]
#county_df_with_fifty_days.describe()

In [7]:
# simple aggregation by state with median AQI by county -- I took the median, but perhaps mean would be better
simple_aggregation_median_AQI = df.groupby(['State','Year'])['Median AQI'].median().reset_index()
simple_aggregation_median_AQI

,State,Year,Median AQI
0,Alabama,1980,47.0
1,Alabama,1981,41.5
2,Alabama,1982,40.5
3,Alabama,1983,44.0
4,Alabama,1984,44.5
...,...,...,...
2281,Wyoming,2020,40.0
2282,Wyoming,2021,41.5
2283,Wyoming,2022,42.0
2284,Wyoming,2023,42.5


In [8]:
simple_aggregation_median_AQI.sample(10)
simple_aggregation_median_AQI.to_csv('gdrive/MyDrive/SIADS_696_Milestone_II/Datasets/AQI_agg.csv',index=False)

OSError: Cannot save file into a non-existent directory: 'gdrive/MyDrive/SIADS_696_Milestone_II/Datasets'

In [ ]:
# filters to only look at days with AQI and moderate days columns
# these and/or other filters should be moved into a function once we finalize which column we want to use as our target
df = df[['State','County','Year','Days with AQI','Moderate Days']]

# calculates the percentage moderate days
df['Percent Moderate Days'] = df['Moderate Days'] / df['Days with AQI']
df.describe()

In [ ]:
# filter dataframe to only include counties with at least 50 days with AQI that year to remove small sample sizes
county_df_with_fifty_days = df[df['Days with AQI'] >= 50]
county_df_with_fifty_days.describe()

In [ ]:
# simple aggregation by state of mean percent moderate days
simple_aggregation_percent_moderate = county_df_with_fifty_days.groupby(['State','Year'])['Percent Moderate Days'].mean().reset_index()
simple_aggregation_percent_moderate.describe()

In [ ]:
simple_aggregation_percent_moderate.sample(10)

In [ ]:
df[df['State'] == 'South Dakota'].Year.unique()

In [ ]:
df['Days with AQI'].unique()